> ## SOLUTION / ANSWER KEY &mdash; Lab 6.10
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-10-external-apis.ipynb`](../lab-10-external-apis.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 6.10 &mdash; Connect to External APIs (Search + Wolfram)

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 3 &middot; Module 6 &mdash; Frameworks for Building AI Agents**

### What you'll do
- Wrap a Google-Search-style tool over a live fact source
- Wrap a Wolfram-Alpha-style exact-compute tool
- Chain them in the executor; optionally call the REAL APIs

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** the graded steps use a tiny **LangChain-shaped shim** (same names &amp; shapes as real LangChain) so they run offline &amp; deterministically. Advanced labs end with an **optional** cell that runs the **real** library.

**Reference:** [Module 6 slides &mdash; Connecting to real tools & APIs](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 6 labs](./index.html)

In [ ]:
# Setup -- run me first
import os
WORK = "/tmp/biaa-lab-06-10"
os.makedirs(WORK, exist_ok=True)
print("Working dir:", WORK)

## Concept
Real agents reach the world through **tool integrations** (deck slide 16): **Google Search** for live
facts beyond the training cutoff, **Wolfram Alpha** for exact computation. The pattern is always: get
a key &rarr; wrap the service as a `@tool` &rarr; add it to the tools list. The **graded** cells use a
deterministic local stand-in; the **optional** cell calls the real APIs if you have keys.

In [ ]:
import ast, operator
# safe arithmetic: walk a parsed AST of numbers + (+ - * / ** unary-minus) -- never bare eval()
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.USub: operator.neg, ast.Pow: operator.pow}
def safe_calc(expr):
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp):
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp):
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError("unsupported expression")
    return ev(ast.parse(expr, mode="eval").body)

# --- LangChain-SHAPED shim: a tool has .name, .description (from the docstring), .args, .invoke() ---
import inspect
class Tool:
    def __init__(self, fn, name=None, description=None):
        self.fn = fn
        self.name = name or fn.__name__
        self.description = (description or inspect.getdoc(fn) or "").strip()
        self.args = list(inspect.signature(fn).parameters)
    def invoke(self, value):
        return self.fn(**value) if isinstance(value, dict) else self.fn(value)
    def __repr__(self):
        return "Tool(name=%r)" % self.name
def tool(fn):
    """The @tool decorator -- wrap a plain function into a Tool (mirrors langchain_core.tools.tool)."""
    return Tool(fn)

class AIMessage:
    def __init__(self, content): self.content = content
class FakeChatModel:
    """Deterministic stand-in for ChatOllama / ChatGroq: replays a scripted list of replies.
    Real code: from langchain_ollama import ChatOllama; model = ChatOllama(model="llama3.2:1b").
    Like the real thing, .invoke(prompt) returns a message whose text is in .content."""
    def __init__(self, script): self.script = list(script); self.i = 0
    def invoke(self, prompt):
        reply = self.script[min(self.i, len(self.script) - 1)]; self.i += 1
        return AIMessage(reply)

class PromptTemplate:
    """Mirrors LangChain: PromptTemplate.from_template(...).format(input=..., ...)."""
    def __init__(self, template): self.template = template
    @classmethod
    def from_template(cls, template): return cls(template)
    def format(self, **kw):
        s = self.template
        for k, v in kw.items():
            s = s.replace("{" + k + "}", str(v))
        return s

def create_react_agent(model, tools, prompt):
    """Bind model + tools + prompt into a ReAct agent (mirrors langchain.agents.create_react_agent)."""
    return {"model": model, "tools": {t.name: t for t in tools}, "prompt": prompt}
def parse_react(text):
    """Turn the model's ReAct text into ('final', answer) or ('action', name, input)."""
    action = inp = None
    for line in text.splitlines():
        s = line.strip()
        if s.startswith("Final Answer:"): return ("final", s.split(":", 1)[1].strip())
        if s.startswith("Action Input:"): inp = s.split(":", 1)[1].strip()
        elif s.startswith("Action:"):      action = s.split(":", 1)[1].strip()
    return ("action", action, inp)
class AgentExecutor:
    """Runs the loop: ask model -> parse -> run tool -> observe -> repeat, capped by max_iterations
    (mirrors langchain.agents.AgentExecutor). verbose=True prints the ReAct trace."""
    def __init__(self, agent, tools=None, verbose=False, max_iterations=6):
        self.agent = agent
        self.tools = agent["tools"] if tools is None else {t.name: t for t in tools}
        self.model = agent["model"]; self.prompt = agent["prompt"]
        self.verbose = verbose; self.max_iterations = max_iterations
    def invoke(self, inputs):
        scratch, steps = "", []
        for _ in range(self.max_iterations):
            text = self.model.invoke(self.prompt.format(input=inputs["input"], scratchpad=scratch)).content
            if self.verbose: print(text)
            parsed = parse_react(text)
            if parsed[0] == "final":
                return {"output": parsed[1], "intermediate_steps": steps}
            name, arg = parsed[1], parsed[2]
            obs = self.tools[name].invoke(arg) if name in self.tools else ("unknown tool: %s" % name)
            if self.verbose: print("Observation:", obs)
            steps.append((name, arg, obs)); scratch += text + "\nObservation: " + str(obs) + "\n"
        return {"output": None, "intermediate_steps": steps}

print("shim ready -- now wrap two external-service tools below")

## Your Turn
Wrap a **google_search** tool (over a small live-fact index) and a **wolfram** compute tool, then let
the executor chain them.

In [ ]:
@tool
def google_search(query):
    """Search the web for a current fact or figure. Use for anything not in the model own memory."""
    index = {"gold price today usd per oz": "2400", "eiffel tower height m": "330"}
    return index.get(query.lower().strip(), "no result")

@tool
def wolfram(expression):
    """Compute an exact math expression (a Wolfram-Alpha-style compute tool)."""
    return safe_calc(expression)

SCRIPT = [
    "Thought: I need the live price.\nAction: google_search\nAction Input: gold price today usd per oz",
    "Thought: double it.\nAction: wolfram\nAction Input: 2400*2",
    "Thought: done.\nFinal Answer: 4800",
]

def make_executor(max_iterations=6):
    model = FakeChatModel(SCRIPT)
    prompt = PromptTemplate.from_template("Q: {input}\n{scratchpad}")
    agent = create_react_agent(model, [google_search, wolfram], prompt)
    return AgentExecutor(agent, max_iterations=max_iterations)

try:
    print('search known  :', google_search.invoke('gold price today usd per oz'))
    print('search unknown:', google_search.invoke('who won the 3pm race'))
    result = make_executor().invoke({"input": "double the current gold price"})
    print('output:', result['output'], '| tools:', [s[0] for s in result['intermediate_steps']])
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("search returns a known live fact", lambda: google_search.invoke("gold price today usd per oz") == "2400")
expect_true("search returns 'no result' for the unknown", lambda: google_search.invoke("who won the 3pm race") == "no result")
expect_true("the compute tool does exact math", lambda: wolfram.invoke("2400*2") == 4800)
expect_true("the executor chains search then compute", lambda: [s[0] for s in make_executor().invoke({"input": "x"})["intermediate_steps"]] == ["google_search", "wolfram"])
expect_true("it reaches the right final answer", lambda: make_executor().invoke({"input": "x"})["output"] == "4800")

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

## Optional &mdash; run this against the REAL LangChain (not graded)
Swap the two stand-in tools for the REAL Google Serper and Wolfram Alpha wrappers. Safe to skip &mdash; it needs `pip install langchain langchain-ollama` (then
`ollama run llama3.2:1b`) or `langchain-groq` with a `GROQ_API_KEY`; external-API tools also need
their own keys. If a package, model or key is missing the cell prints a friendly note and moves on.
**The graded steps above use a deterministic LangChain-shaped shim, so the lab always verifies offline.**

In [ ]:
import os
try:
    from langchain_community.utilities import GoogleSerperAPIWrapper
    if os.getenv("SERPER_API_KEY"):
        print("Google (Serper):", GoogleSerperAPIWrapper().run("current gold price per ounce"))
    else:
        print("Set SERPER_API_KEY (serper.dev) to run a REAL Google search here -- skipping.")
except Exception as e:
    print("Install langchain-community for the real search wrapper -- skipping:", type(e).__name__)
try:
    from langchain_community.utilities.wolfram_alpha import WolframAlphaAPIWrapper
    if os.getenv("WOLFRAM_ALPHA_APPID"):
        print("Wolfram Alpha:", WolframAlphaAPIWrapper().run("2400 * 2"))
    else:
        print("Set WOLFRAM_ALPHA_APPID (developer.wolframalpha.com) for real compute -- skipping.")
except Exception as e:
    print("Install langchain-community + wolframalpha for the real wrapper -- skipping:", type(e).__name__)
print("The graded tools above already ran offline against a local stand-in.")

---
### Key takeaway
Get a key -> wrap the service as a @tool -> add it to the list. That is how an agent reaches live facts and real computation -- the client's Day-3 external-API lab, for real.

[&#8592; All Module 6 labs](./index.html) &nbsp;&middot;&nbsp; [Module 6 slides](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>